# Otimização por Enxame de Partículas (PSO)

O **PSO** (*Particle Swarm Optimization*) é uma **metaheurística bioinspirada**
proposta por Kennedy e Eberhart (1995), que imita o comportamento social de
**bandos de pássaros** e **cardumes de peixes** na busca por alimento.

## Ideia geral

Um conjunto de **partículas** (soluções candidatas) "voa" pelo espaço de busca.
Cada partícula $i$ possui:

- uma **posição** $\vec{x}_i$ — uma solução candidata;
- uma **velocidade** $\vec{v}_i$ — a direção e o passo do próximo movimento;
- a sua **melhor posição individual** $\vec{p}_i$ (*pbest*);
- o conhecimento da **melhor posição global** $\vec{g}$ (*gbest*) de todo o enxame.

## Equações de atualização

A cada iteração, a velocidade e a posição são atualizadas por:

$$
\vec{v}_i \leftarrow w\,\vec{v}_i
\;+\; c_1 r_1 (\vec{p}_i - \vec{x}_i)
\;+\; c_2 r_2 (\vec{g} - \vec{x}_i)
$$

$$
\vec{x}_i \leftarrow \vec{x}_i + \vec{v}_i
$$

onde:

| Símbolo | Significado |
|---------|-------------|
| $w$ | **inércia** — controla a exploração; aqui decai linearmente de $w_{ini}$ a $w_{fin}$ |
| $c_1$ | coeficiente **cognitivo** — atração à melhor posição individual |
| $c_2$ | coeficiente **social** — atração à melhor posição global |
| $r_1, r_2$ | números aleatórios em $[0,1]$ que dão estocasticidade à busca |

O termo cognitivo puxa a partícula para a sua própria experiência; o termo social
puxa para a melhor solução já encontrada pelo grupo. O equilíbrio entre **exploração**
(inércia alta) e **explotação** (inércia baixa) é a chave do método.

## Restrições e o método da penalidade

O PSO é, por natureza, um otimizador **sem restrições**. Para resolver o
**Problema de Programação Linear** (2º script) usamos o **método da penalidade**:
toda violação de uma restrição $g(\vec{x}) \le 0$ subtrai um valor grande da função
objetivo, $f(\vec{x}) - \lambda \sum \max(0, g(\vec{x}))^2$, empurrando as partículas
de volta para a **região viável**.

> **Nos scripts:** o 1º minimiza $f(x,y) = x^2 + y^2 + 2$ (ótimo em $(0,0)$, $f=2$);
> o 2º maximiza o lucro $Z = 5x + 4y$ sujeito a restrições de produção.

**PSO — Minimização de f(x,y) = x² + y² + 2**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Função objetivo ──────────────────────────────────────────────────────────
def f(x, y):
    return x**2 + y**2 + 2

# ── Parâmetros do PSO ────────────────────────────────────────────────────────
np.random.seed(42)
n     = 20       # partículas
m     = 40       # iterações
w_ini = 0.9      # inércia inicial
w_fin = 0.2      # inércia final
c1 = c2 = 1.5    # coeficientes cognitivo / social
LIM   = 5.0      # domínio [-LIM, LIM]

# ── Inicialização ────────────────────────────────────────────────────────────
x  = np.random.uniform(-LIM, LIM, (n, 2))      # posições
v  = np.zeros((n, 2))                           # velocidades
pb = x.copy()                                   # melhor posição individual
fb = np.array([f(xi[0], xi[1]) for xi in x])   # melhor valor individual
gb_idx = np.argmin(fb)
gb  = pb[gb_idx].copy()  # melhor global
fgb = fb[gb_idx]

# ── Histórico para animação ──────────────────────────────────────────────────
hist_x  = [x.copy()]
hist_gb = [fgb]

# ── Loop principal PSO ───────────────────────────────────────────────────────
for i in range(m):
    w  = w_ini + (w_fin - w_ini) * i / (m - 1)
    r1 = np.random.rand(n, 2)
    r2 = np.random.rand(n, 2)
    v  = w * v + c1 * r1 * (pb - x) + c2 * r2 * (gb - x)
    x  = x + v
    # Clampar no domínio (prender as partículas dentro do domínio)
    x  = np.clip(x, -LIM, LIM)
    # Atualizar melhores individuais e global
    for k in range(n):
        fk = f(x[k, 0], x[k, 1])
        if fk < fb[k]:
            fb[k] = fk
            pb[k] = x[k].copy()
        if fk < fgb:
            fgb = fk
            gb  = x[k].copy()
    hist_x.append(x.copy())
    hist_gb.append(fgb)

# ── Figura final (2 painéis) ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor="#0d1117")
for ax in axes:
    ax.set_facecolor("#0d1117")

# ── Painel 1 — curvas de nível + enxame no último estado ────────────────────
ax1 = axes[0]
gx = np.linspace(-LIM, LIM, 300)
gy = np.linspace(-LIM, LIM, 300)
X, Y = np.meshgrid(gx, gy)
Z = f(X, Y)

cp = ax1.contourf(X, Y, Z, levels=30, cmap="plasma", alpha=0.85)
plt.colorbar(cp, ax=ax1, label="f(x,y)")
ax1.contour(X, Y, Z, levels=30, colors="white", linewidths=0.3, alpha=0.3)

ax1.scatter(hist_x[0][:, 0], hist_x[0][:, 1],
            c="cyan", s=50, zorder=3, alpha=0.6, label="Posição inicial")
ax1.scatter(hist_x[-1][:, 0], hist_x[-1][:, 1],
            c="lime", s=50, zorder=4, alpha=0.9, label="Posição final")
ax1.scatter(*gb, c="red", s=200, zorder=5, marker="*",
            label=f"Ótimo global\n({gb[0]:.4f}, {gb[1]:.4f})")

ax1.set_xlim(-LIM, LIM)
ax1.set_ylim(-LIM, LIM)
ax1.set_xlabel("x", color="white")
ax1.set_ylabel("y", color="white")
ax1.set_title("Enxame de Partículas — Espaço de Busca", color="white", fontsize=12)
ax1.tick_params(colors="white")
for spine in ax1.spines.values():
    spine.set_edgecolor("gray")
ax1.legend(facecolor="#1a1f2e", edgecolor="gray", labelcolor="white", fontsize=9)

# ── Painel 2 — convergência do melhor global ─────────────────────────────────
ax2 = axes[1]
iters = range(m + 1)
ax2.plot(iters, hist_gb, color="#ff6b6b", linewidth=2.5, label="Melhor global f*")
ax2.axhline(y=2.0, color="lime", linewidth=1.5, linestyle="--", label="Mínimo teórico = 2")
ax2.fill_between(iters, hist_gb, 2.0, alpha=0.15, color="#ff6b6b")

ax2.set_xlabel("Iteração", color="white")
ax2.set_ylabel("f*(iteração)", color="white")
ax2.set_title("Convergência do PSO", color="white", fontsize=12)
ax2.tick_params(colors="white")
ax2.set_facecolor("#0d1117")
for spine in ax2.spines.values():
    spine.set_edgecolor("gray")
ax2.legend(facecolor="#1a1f2e", edgecolor="gray", labelcolor="white", fontsize=10)

ax2.text(m * 0.55, hist_gb[0] * 0.97,
         f"f*final = {fgb:.6f}\nx* = ({gb[0]:.4f}, {gb[1]:.4f})",
         color="white", fontsize=10,
         bbox=dict(boxstyle="round", facecolor="#1a1f2e", edgecolor="gray"))

plt.suptitle("PSO — Minimização de f(x,y) = x² + y² + 2", color="white", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("pso_resultado.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.close()

print(f"Mínimo encontrado : f = {fgb:.8f}")
print(f"Ponto ótimo       : x = {gb[0]:.8f}, y = {gb[1]:.8f}")
print(f"Mínimo teórico    : f = 2.0 em (0, 0)")


**PSO — Programação Linear (Maximizar Z = 5x + 4y)**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ════════════════════════════════════════════════════════════════════════════
# PROBLEMA DE PROGRAMAÇÃO LINEAR
# ════════════════════════════════════════════════════════════════════════════
#
# Uma fábrica produz dois produtos A e B:
#
# Maximizar:    Z = 5x + 4y   (lucro em R$)
#
# Sujeito a:
#   Restrição 1 (horas de máquina):      6x + 4y ≤ 24
#   Restrição 2 (horas de mão de obra):   x + 2y ≤ 6
#   x ≥ 0, y ≥ 0
#
# Solução analítica (vértice ótimo): x=3, y=1.5 → Z = 21
# ════════════════════════════════════════════════════════════════════════════

# ── Função objetivo (maximizar Z = 5x + 4y) ─────────────────────────────────
def objetivo(x, y):
    return 5*x + 4*y

# ── Penalidade por violação das restrições ───────────────────────────────────
PENALIDADE = 1e6

def f(x, y):
    z = objetivo(x, y)
    violacao = 0.0
    # Restrição 1: 6x + 4y <= 24
    if 6*x + 4*y > 24:
        violacao += (6*x + 4*y - 24) ** 2
    # Restrição 2: x + 2y <= 6
    if x + 2*y > 6:
        violacao += (x + 2*y - 6) ** 2
    # Não negatividade (x >= 0, y >= 0)
    if x < 0:
        violacao += x ** 2
    if y < 0:
        violacao += y ** 2
    return z - PENALIDADE * violacao  # maximizar → penaliza violações

# ── Parâmetros do PSO ────────────────────────────────────────────────────────
np.random.seed(7)
n     = 30    # partículas
m     = 60    # iterações
w_ini = 0.9
w_fin = 0.2
c1 = c2 = 1.5
LIM   = 6.0   # domínio [0, LIM] para x e y

# ── Inicialização ────────────────────────────────────────────────────────────
x  = np.random.uniform(0, LIM, (n, 2))
v  = np.zeros((n, 2))
pb = x.copy()
fb = np.array([f(xi[0], xi[1]) for xi in x])
gb_idx = np.argmax(fb)  # MÁXIMO
gb  = pb[gb_idx].copy()
fgb = fb[gb_idx]

hist_x  = [x.copy()]
hist_gb = [objetivo(gb[0], gb[1])]  # guarda o Z sem penalidade

# ── Loop principal PSO ───────────────────────────────────────────────────────
for i in range(m):
    w  = w_ini + (w_fin - w_ini) * i / (m - 1)
    r1 = np.random.rand(n, 2)
    r2 = np.random.rand(n, 2)
    v  = w * v + c1 * r1 * (pb - x) + c2 * r2 * (gb - x)
    x  = x + v
    x  = np.clip(x, 0, LIM)  # mantém x,y >= 0
    for k in range(n):
        fk = f(x[k, 0], x[k, 1])
        if fk > fb[k]:  # MAXIMIZAR
            fb[k] = fk
            pb[k] = x[k].copy()
        if fk > fgb:
            fgb = fk
            gb  = x[k].copy()
    hist_x.append(x.copy())
    hist_gb.append(objetivo(gb[0], gb[1]))

x_opt, y_opt = gb[0], gb[1]
z_opt = objetivo(x_opt, y_opt)

print("=" * 45)
print("  RESULTADO DO PSO")
print("=" * 45)
print(f"  x (Produto A) = {x_opt:.6f}")
print(f"  y (Produto B) = {y_opt:.6f}")
print(f"  Z (Lucro)     = R$ {z_opt:.4f}")
print("-" * 45)
print(f"  Restr. 1: 6x+4y = {6*x_opt+4*y_opt:.4f} ≤ 24  {'✓' if 6*x_opt+4*y_opt <= 24.01 else '✗'}")
print(f"  Restr. 2: x+2y  = {x_opt+2*y_opt:.4f}  ≤ 6   {'✓' if x_opt+2*y_opt <= 6.01 else '✗'}")
print("=" * 45)
print(f"  Solução analítica: x=3, y=1.5, Z=21")
print("=" * 45)

# ════════════════════════════════════════════════════════════════════════════
# GRÁFICOS
# ════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor="#0d1117")
for ax in axes:
    ax.set_facecolor("#0d1117")

# ── Painel 1 — Região viável + enxame ───────────────────────────────────────
ax1 = axes[0]
gx = np.linspace(0, LIM, 400)
gy = np.linspace(0, LIM, 400)
X, Y = np.meshgrid(gx, gy)
Z = 5*X + 4*Y

# Região viável (ambas restrições satisfeitas)
viavel = (6*X + 4*Y <= 24) & (X + 2*Y <= 6)
Z_plot = np.where(viavel, Z, np.nan)

cp = ax1.contourf(X, Y, Z_plot, levels=20, cmap="YlOrRd", alpha=0.85)
plt.colorbar(cp, ax=ax1, label="Z = 5x + 4y (região viável)")

# Fronteiras das restrições
x_r = np.linspace(0, LIM, 300)
ax1.plot(x_r, (24 - 6*x_r) / 4, color="cyan", lw=2, label="6x + 4y = 24")
ax1.plot(x_r, (6 - x_r) / 2,    color="lime", lw=2, label="x + 2y = 6")
ax1.axhline(0, color="white", lw=0.8, alpha=0.4)
ax1.axvline(0, color="white", lw=0.8, alpha=0.4)

# Vértices da região viável
vertices = [(0, 0), (4, 0), (3, 1.5), (0, 3)]
for vx, vy in vertices:
    ax1.plot(vx, vy, "o", color="white", ms=7, zorder=5)
    ax1.annotate(f"({vx}, {vy})\nZ={5*vx+4*vy}",
                 xy=(vx, vy), xytext=(vx+0.15, vy+0.15),
                 color="white", fontsize=8)

# Posições do enxame
ax1.scatter(hist_x[0][:, 0], hist_x[0][:, 1],
            c="deepskyblue", s=35, alpha=0.5, zorder=3, label="Posição inicial")
ax1.scatter(hist_x[-1][:, 0], hist_x[-1][:, 1],
            c="limegreen", s=35, alpha=0.8, zorder=4, label="Posição final")
ax1.scatter(x_opt, y_opt, c="red", s=250, marker="*", zorder=6,
            label=f"Ótimo PSO\n({x_opt:.3f}, {y_opt:.3f})\nZ={z_opt:.3f}")

ax1.set_xlim(0, LIM)
ax1.set_ylim(0, LIM)
ax1.set_xlabel("x (Produto A)", color="white", fontsize=11)
ax1.set_ylabel("y (Produto B)", color="white", fontsize=11)
ax1.set_title("PSO — Espaço de Busca e Região Viável", color="white", fontsize=12)
ax1.tick_params(colors="white")
for sp in ax1.spines.values():
    sp.set_edgecolor("gray")
ax1.legend(facecolor="#1a1f2e", edgecolor="gray", labelcolor="white", fontsize=8, loc="upper right")

# ── Painel 2 — Convergência de Z ────────────────────────────────────────────
ax2 = axes[1]
iters = range(m + 1)
ax2.plot(iters, hist_gb, color="#ff6b6b", lw=2.5, label="Melhor Z (PSO)")
ax2.axhline(y=21, color="lime", lw=1.5, ls="--", label="Ótimo analítico Z = 21")
ax2.fill_between(iters, hist_gb, 21, alpha=0.12, color="#ff6b6b")

ax2.set_xlabel("Iteração", color="white", fontsize=11)
ax2.set_ylabel("Z = 5x + 4y", color="white", fontsize=11)
ax2.set_title("Convergência do Valor Ótimo", color="white", fontsize=12)
ax2.tick_params(colors="white")
ax2.set_facecolor("#0d1117")
for sp in ax2.spines.values():
    sp.set_edgecolor("gray")
ax2.legend(facecolor="#1a1f2e", edgecolor="gray", labelcolor="white", fontsize=10)

ax2.text(m * 0.45, min(hist_gb) * 1.02,
         f"Z* = {z_opt:.4f}\nx* = {x_opt:.4f}\ny* = {y_opt:.4f}",
         color="white", fontsize=10,
         bbox=dict(boxstyle="round", facecolor="#1a1f2e", edgecolor="gray"))

plt.suptitle(
    "PSO — Programação Linear: Maximizar Z = 5x + 4y\n"
    "Sujeito a: 6x + 4y ≤ 24 | x + 2y ≤ 6 | x, y ≥ 0",
    color="white", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("pso_ppl.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
